# 04 — Model Architecture and Training

**Purpose:** Walk through the U-Net DDPM architecture and training configuration.

This notebook documents the model design and training setup without requiring a full
training run. It covers:

1. **Model definition** — instantiates the `Unet` with `dim=64`,
   `dim_mults=(1, 2, 4, 8)`, `channels=2`, and `flash_attn=True`, giving 35.7 M
   trainable parameters across four encoder/decoder stages. Wraps it in
   `GaussianDiffusion` with 1000 timesteps, sigmoid noise schedule, and v-prediction
   objective.

2. **Data loading and augmentation** — loads the stacked CIB+tSZ `.npy` arrays,
   performs the 80/20 train/validation split, and applies the 8× augmentation
   (4 rotations × horizontal flip) via `augment_images_unique`.

3. **Trainer configuration** — explains each `Trainer1D` hyperparameter: batch size 16,
   learning rate 1×10⁻⁴, 100,000 steps, gradient accumulation every 2 steps,
   EMA decay 0.995, mixed precision (fp16), checkpoint every 5,000 steps.

To actually train, run `accelerate launch foregrounds_diffusion/train.py` from the repo
root instead of executing this notebook end-to-end.

**Inputs:**
- CIB patches: `data/low_pass/2mJy/CIB_map_150GHz_256_st6_minmax_2mJy_zero_lp.npy`
- tSZ patches: `data/low_pass/2mJy/tSZ3_map_150GHz_256_st6_minmax_2mJy_norm_lp.npy`

**Outputs:** model graph, parameter count table (no checkpoint written).

**Key module functions:** none — uses `denoising_diffusion_pytorch` directly.

**Paper reference:** §3.1 (DDPM framework), Appendix A (Table 1 — architecture details).

In [1]:
import numpy as np
import torch
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer1D, Dataset1D

PTSRC = 2
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cpu


In [2]:
unet = Unet(
    dim=64,
    dim_mults=(1, 2, 4, 8),
    channels=2,          # CIB + tSZ
    flash_attn=True,
)

diffusion = GaussianDiffusion(
    unet,
    image_size=256,
    timesteps=1000,      # T = 1000 diffusion steps
)
diffusion = diffusion.to(device)

# Parameter count (paper reports 35.7 M)
total_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}  ({total_params / 1e6:.1f} M)")


Trainable parameters: 35,708,290  (35.7 M)


In [3]:
fpath_cib = f"data/low_pass/{PTSRC}mJy/CIB_map_150GHz_256_st6_minmax_{PTSRC}mJy_zero_lp.npy"
fpath_tsz = f"data/low_pass/{PTSRC}mJy/tSZ3_map_150GHz_256_st6_minmax_{PTSRC}mJy_norm_lp.npy"

cib_maps = np.load(fpath_cib)  # (N, H, W, 1)
tsz_maps = np.load(fpath_tsz)  # (N, H, W, 1)
cut_maps = np.concatenate([cib_maps, tsz_maps], axis=-1)  # (N, H, W, 2)
cut_maps = cut_maps.transpose(0, 3, 1, 2)                 # (N, 2, H, W)
print(f"Stacked patches: {cut_maps.shape}")

# 80 / 20 train / validation split (seeded for reproducibility)
rng = np.random.default_rng(seed=42)
indices  = rng.permutation(len(cut_maps))
num_train = int(0.8 * len(cut_maps))
training_images = torch.tensor(cut_maps[indices[:num_train]], dtype=torch.float32)
val_images      = torch.tensor(cut_maps[indices[num_train:]], dtype=torch.float32)
print(f"Train: {len(training_images)},  Val: {len(val_images)}")


Stacked patches: (674, 2, 256, 256)
Train: 539,  Val: 135


In [4]:
def augment_images_unique(images):
    """8× augmentation: 4 rotations × horizontal flip."""
    augmented = []
    for img in images:
        for k in range(4):
            rotated = torch.rot90(img, k=k, dims=(1, 2))
            augmented.append(rotated)
            augmented.append(torch.flip(rotated, dims=[2]))
    return torch.stack(augmented)

augmented = augment_images_unique(training_images)
print(f"After 8× augmentation: {len(training_images)} → {len(augmented)} training samples")


After 8× augmentation: 539 → 4312 training samples


In [5]:
dataset = Dataset1D(augmented)

trainer = Trainer1D(
    diffusion,
    dataset=dataset,
    train_batch_size=16,
    num_samples=1,
    train_lr=1e-4,
    train_num_steps=100_000,
    save_and_sample_every=5_000,
    gradient_accumulate_every=2,    # effective batch size = 16 × 2 = 32
    ema_decay=0.995,
    amp=True,                       # fp16 mixed precision
)
print("Trainer configured.")
print("To train:  accelerate launch foregrounds_diffusion/train.py")
print("To resume: trainer.load(<step_number>)")


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer configured.
To train:  accelerate launch foregrounds_diffusion/train.py
To resume: trainer.load(<step_number>)
